In [1]:
import Pkg;
Pkg.add("StatsBase")

using LinearAlgebra
using StatsBase

   Resolving package versions...
  No Changes to `C:\Users\alex1\.julia\environments\v1.11\Project.toml`
  No Changes to `C:\Users\alex1\.julia\environments\v1.11\Manifest.toml`


In [2]:
using Random, StatsBase

Random.seed!(1234)  # set the global seed

TaskLocalRNG()

# General
This code implements the Approximate Cholesky algorithm in the most naive way. It uses dense matrix operations. We have optimized this for readability. For a full solve, this approximate factorization is then used with PCG to solve a system.

A fast sparse solver (that also implements the full pipeline) is publicly available here:
[Github link: SDD solver](https://rjkyng.github.io/SDDM2023/)

Attached in the email you can see comparisons between the optimized solver and other incomplete cholesky implementations or Hypres BoomerAMG on problems up to 10^9 nonzeros.

## Disclaimer
We have not properly tested this notebook and there might be errors in the implementation. There can also be stability issues depending on the randomness in this implementation. This is not the case in the serious implementation. For further benchmarking please use the public solver above.


In [3]:
# This is general Cholesky

function myChol(L)
  n = size(L)[1]
  S = copy(L)
  fac = zeros(n,n-1)
  for i = 1:(n-1)
    li = S[:,i]/S[i,i]^(1/2)
    fac[:,i] = li
    S = S - li*li'
  end
  return fac
end

myChol (generic function with 1 method)

In [4]:
# Consider a clique on n vertices
n = 10
L_Kn = I - ones(n,n)/n

chol_fac = myChol(L_Kn) # Generate Factor

@show opnorm(chol_fac - tril(chol_fac)) # Check that the factorization is lower triangular

@show opnorm(L_Kn-chol_fac*chol_fac') # Check that the factorization equals the input matrix

chol_fac

opnorm(chol_fac - tril(chol_fac)) = 3.0195616480790007e-16
opnorm(L_Kn - chol_fac * chol_fac') = 3.893763993344898e-16


10×9 Matrix{Float64}:
  0.948683   1.47196e-17   1.66905e-17  …   5.09902e-17   8.83177e-17
 -0.105409   0.942809      1.4836e-17       4.53247e-17   7.85046e-17
 -0.105409  -0.117851      0.935414        -7.93182e-17  -1.37383e-16
 -0.105409  -0.117851     -0.133631        -6.7987e-17   -1.17757e-16
 -0.105409  -0.117851     -0.133631         0.0           0.0
 -0.105409  -0.117851     -0.133631     …   4.53247e-17   7.85046e-17
 -0.105409  -0.117851     -0.133631         3.39935e-17   5.88785e-17
 -0.105409  -0.117851     -0.133631         0.816497      0.0
 -0.105409  -0.117851     -0.133631        -0.408248      0.707107
 -0.105409  -0.117851     -0.133631        -0.408248     -0.707107

In [5]:
# This is still normal cholesky, we just rewrite how we update the schur complement
function myChol2(L)
  n = size(L)[1]
  S = copy(L)
  fac = zeros(n,n-1)
  for i = 1:(n-1)
    li = S[:,i]/S[i,i]^(1/2)
    fac[:,i] = li
    # Same up to here

    # Remove the star...
    ai = -S[:,i]
    di = S[i,i]
    ai[i] = 0

    S_noi = copy(S)
    S_noi[:,i] .= 0
    S_noi[i,:] .= 0
    S_noi -= Diagonal(ai)

    S = S_noi + Diagonal(ai)-ai*ai'/di # ...and insert the clique

  end
  return fac
end

myChol2 (generic function with 1 method)

In [6]:
# Same example with the second, equivalent Cholesky code
n = 10
L_Kn = I - ones(n,n)/n

chol_fac = myChol2(L_Kn) # Generate Factor

@show opnorm(chol_fac - tril(chol_fac)) # Check that the factorization is lower triangular

@show opnorm(L_Kn-chol_fac*chol_fac') # Check that the factorization equals the input matrix

chol_fac

opnorm(chol_fac - tril(chol_fac)) = 0.0
opnorm(L_Kn - chol_fac * chol_fac') = 6.794445557167506e-16


10×9 Matrix{Float64}:
  0.948683   0.0        0.0       …   0.0        0.0        0.0
 -0.105409   0.942809   0.0           0.0        0.0        0.0
 -0.105409  -0.117851   0.935414      0.0        0.0        0.0
 -0.105409  -0.117851  -0.133631      0.0        0.0        0.0
 -0.105409  -0.117851  -0.133631      0.0        0.0        0.0
 -0.105409  -0.117851  -0.133631  …   0.0        0.0        0.0
 -0.105409  -0.117851  -0.133631      0.866025   0.0        0.0
 -0.105409  -0.117851  -0.133631     -0.288675   0.816497   0.0
 -0.105409  -0.117851  -0.133631     -0.288675  -0.408248   0.707107
 -0.105409  -0.117851  -0.133631     -0.288675  -0.408248  -0.707107

In [7]:
# Now the simple aprox-chol:

# Sampling rule for a single edge
function cliqueSample(a)
  nbrs = findall(a .> 1e-12)
  cliqueSampleLap = zeros(n,n)
  for j in nbrs
    k = sample(1:n,Weights(a))
    if k != j
      wjk = a[j]*a[k]/(a[j]+a[k])
      bjk = zeros(n); bjk[j] = 1; bjk[k] = -1;
      cliqueSampleLap += wjk*bjk*bjk'
    end
  end
  return cliqueSampleLap
end

cliqueSample (generic function with 1 method)

In [ ]:
function cliqueSample_sdd(a,i)
  total_weight= a[i]
  self_loop= 2*a[i]-sum(abs,a)
  a[i]=0
  nbrs = findall(abs.(a) .> 1e-12)
  cliqueSampleLap = zeros(n,n)
  cliqueSampleLap[diagind(cliqueSampleLap)] .+= self_loop* abs.(a)/total_weight
  a[i]= self_loop
  for j in nbrs
    k = sample(1:n,Weights(abs.(a)))
    if (k != j) & (j != i) &  (k != i)
      wjk = abs(a[j]*a[k])/(abs(a[j])+ abs(a[k]))
      bjk = zeros(n); bjk[j] = 1; bjk[k] = -sign(a[k]* a[j]);
      cliqueSampleLap += wjk*bjk*bjk'
    end
  end
  return cliqueSampleLap
end

cliqueSample_sdd (generic function with 1 method)

In [ ]:
# This is the excat same algorithm as just with sampling
function apxChol1(L)
  n = size(L)[1]
  S = copy(L)
  fac = zeros(n,n-1)
  connected = true
  for i = 1:(n-1)
    if S[i,i] < 1e-12 # If the diagonal is zero, then the row is already zeroed out
      if connected
        println("The factorization disconnected the graph, anything can happen now!")
        connected = false
      end

      continue
    end

    li = S[:,i]/S[i,i]^(1/2)
    fac[:,i] = li

    # Remove the star...
    ai = -S[:,i]
    di = S[i,i]
    ai[i] = 0

    S_noi = copy(S)
    S_noi[:,i] .= 0
    S_noi[i,:] .= 0
    S_noi -= Diagonal(ai)
    if i==n-1
      

    S = S_noi + cliqueSample(ai) # Only difference: "Diagonal(ai)-ai*ai'/di" -> "cliqueSample(ai)"
  end
  return fac
end

apxChol1 (generic function with 1 method)

In [10]:
function erase_self_loops(A::AbstractMatrix, B::AbstractMatrix)
    m, n = size(A)
    @assert size(B) == (m, n) "Matrices A and B must have the same dimensions"

    # Initialize C with zeros (efficient allocation)
    # Use the same type as the input matrices (e.g., Float64)
    C = zeros(promote_type(eltype(A), eltype(B)), m, n)

    @inbounds for j in 1:n
        for i in 1:m
            valA = A[i, j]
            valB = B[i, j]
            if (abs(valA) >1e-12) & (abs(valB) >1e-12)               
                # 1. Base assignment
                C[i, j] = valA + valB
                
                # 2. Calculate the penalty term
                # Term = 2 * (|A| + |B| - |A+B|)
                term =  (abs(valA) + abs(valB) - abs(valA + valB))
                # 3. Add to diagonals
                C[i, i] += term
                C[j, j] += term 
            end
            if (abs(valA) >1e-12) & (abs(valB) <=1e-12)
                C[i,j] = valA
            end
            if (abs(valA) <=1e-12) & (abs(valB) <=1e-12)
                C[i,j] = valB
            end
            # If signs are the same, "nothing happens" (C[i,j] remains 0)
        end
    end

    return C
end

erase_self_loops (generic function with 1 method)

In [ ]:
function apxChol_sdd1(L)
  n = size(L)[1]
  S = copy(L)
  fac = zeros(n,n)
  connected = true
  for i = 1:(n-1)
    if S[i,i] < 1e-12 # If the diagonal is zero, then the row is already zeroed out
      if connected
        println("The factorization disconnected the graph, anything can happen now!")
        connected = false
      end

      continue
    end

    li = S[:,i]/S[i,i]^(1/2)
    fac[:,i] = li

    # Remove the star...
    ai = S[:,i]
    di = S[i,i]
    #ai[i] = 0

    S_noi = copy(S)
    S_noi -=Diagonal(abs.(ai))
    S_noi[:,i] .= 0
    S_noi[i,:] .= 0

    clique_new= cliqueSample_sdd(ai,i)
    if i==n-1 
      fac[n,n] = (abs(S_noi[n,n]+clique_new[n,n]))^(1/2)
    end
    S = erase_self_loops(S_noi,clique_new)
  end
  return fac
end

apxChol_sdd1 (generic function with 1 method)

In [12]:
# Same example
n = 10
L_Kn = I - ones(n,n)/n

fApxChol = apxChol1(L_Kn)
rel_eigs = eigvals((L_Kn + ones(n,n) / n)*inv(fApxChol*fApxChol' + ones(n,n) / n))

# The ratio of the smallest to the largest eigenvalue
maximum(rel_eigs) / minimum(rel_eigs)

2.4775271640742416

In [13]:
# Same example
n = 10
L_Kn = I - ones(n,n)/n
L_Kn[1,2]= -L_Kn[1,2]
L_Kn[2,1]= -L_Kn[2,1]

fApxChol = apxChol_sdd1(L_Kn)
rel_eigs = eigvals((L_Kn + ones(n,n) / n)*inv(fApxChol*fApxChol' + ones(n,n) / n))

# The ratio of the smallest to the largest eigenvalue
maximum(rel_eigs) / minimum(rel_eigs)

1.912175823188673

In [14]:
# Now we allow multiedges for the sampling
function cliqueSampleMultiEdge(a, multiplicity)
  nbrs = findall(a .> 1e-12)
  n = size(a)[1]
  cliqueSampleLap = zeros(n,n)
  new_multiedges = zeros(Int, n,n)
  for j in nbrs
    for rep in 1:multiplicity[j]
      k = sample(1:n,Weights(a))
      if k != j
        multiWeightJ = a[j] / multiplicity[j]
        multiWeightk = a[k]
        wjk =  multiWeightJ * a[k] / (a[j] + a[k])
        new_multiedges[j,k] += 1; new_multiedges[k,j] += 1;
        bjk = zeros(n); bjk[j] = 1; bjk[k] = -1;
        cliqueSampleLap += wjk*bjk*bjk'
      end
    end
  end
  return cliqueSampleLap, new_multiedges
end

cliqueSampleMultiEdge (generic function with 1 method)

In [ ]:
function cliqueSampleMultiEdgeSdd(a, multiplicity,i)
  total_weight= a[i]
  self_loop= 2*a[i]-sum(abs,a)
  a[i]=0
  nbrs = findall(abs.(a) .> 1e-12)
  cliqueSampleLap = zeros(n,n)
  cliqueSampleLap[diagind(cliqueSampleLap)] .+= self_loop/total_weight* abs.(a)
  a[i]= self_loop
  new_multiedges = zeros(Int, n,n)
  for j in nbrs
    for rep in 1:multiplicity[j]
      k = sample(1:n,Weights(abs.(a)))
      if (k != j) & (j != i) &  (k != i)
        multiWeightJ = a[j] / multiplicity[j]
        multiWeightk = a[k]
        wjk =  abs(multiWeightJ * a[k]) / (abs(a[j])+ abs(a[k]))
        new_multiedges[j,k] += 1; new_multiedges[k,j] += 1;
        bjk = zeros(n); bjk[j] = 1; bjk[k] = -sign(a[k]* a[j]);
        cliqueSampleLap += wjk*bjk*bjk'
      end
    end
  end
  return cliqueSampleLap, new_multiedges
end

cliqueSampleMultiEdgeSdd (generic function with 1 method)

In [ ]:
function cliqueSampleMultiEdgeHdd(a, multiplicity,i)
  total_weight= abs(a[i])
  self_loop= 2*abs(a[i])-sum(abs,a)
  a[i]=0
  nbrs = findall(abs.(a) .> 1e-12)
  cliqueSampleLap = zeros(ComplexF64,n,n)
  cliqueSampleLap[diagind(cliqueSampleLap)] .+= self_loop/total_weight* abs.(a)
  a[i]= self_loop
  new_multiedges = zeros(Int, n,n)
  for j in nbrs
    for rep in 1:multiplicity[j]
      k = sample(1:n,Weights(abs.(a)))
      if (k != j) & (j != i) &  (k != i)
        multiWeightJ = a[j] / multiplicity[j]
        multiWeightk = a[k]
        wjk =  abs(multiWeightJ * a[k]) / (abs(a[j])+ abs(a[k]))
        new_multiedges[j,k] += 1; new_multiedges[k,j] += 1;
        bjk = zeros(ComplexF64,n); bjk[j] = 1; bjk[k] = -sign(a[k])/sign(a[j]);
        cliqueSampleLap += wjk*bjk*bjk'
      end
    end
  end
  return cliqueSampleLap, new_multiedges
end

In [ ]:
using LinearAlgebra, StatsBase

function cliqueSampleMultiEdgeHddEff(a::Vector{ComplexF64}, multiplicity::Vector{Int}, i::Int)
    n = length(a)
    
    # 1. Pre-calculate magnitudes to avoid repeated abs() calls
    # We copy 'a' logic without mutating the input vector permanently
    abs_a = abs.(a)
    val_i = a[i]
    abs_val_i = abs_a[i]
    
    # 2. Logic for self_loop and initialization
    # Original logic: self_loop = 2*|a[i]| - sum(|a|)
    # Note: sum(abs_a) includes |a[i]|, so we use that.
    total_weight = abs_val_i
    sum_abs_all = sum(abs_a)
    self_loop = abs(2 * abs_val_i - sum_abs_all)
    #println(self_loop)
    # 3. Initialize Output Matrices
    cliqueSampleLap = zeros(ComplexF64, n, n)
    new_multiedges = zeros(Int, n, n)
    
    # 4. Handle Diagonal Initialization (Vectorized)
    # Corresponds to: cliqueSampleLap[diagind] .+= self_loop/total_weight * abs.(a)
    # In original code, a[i] was set to 0 before this, so index i gets 0 update.
    ratio = self_loop / total_weight
    @inbounds for k in 1:n
        if k != i
            cliqueSampleLap[k, k] = ratio * abs_a[k]
        end
    end

    # 5. Setup Sampling ONCE (Big Performance Win)
    # We mask position i to 0.0 so we never sample it (removes need for k != i check)
    sampling_weights = copy(abs_a)
    sampling_weights[i] = self_loop
    wv = Weights(sampling_weights) # O(N) operation happens only once now
    
    # Identify neighbors (indices where weight > threshold)
    # We use valid indices from abs_a (excluding i effectively because we skip it in loop or weights)
    nbrs = findall(x -> x > 1e-12, sampling_weights)

    # 6. Main Loop
    for j in nbrs
        # Pre-calculate j-specific constants
        val_j = a[j]
        abs_val_j = abs_a[j]
        sign_j = sign(a[j]) #val_j / abs_val_j # Equivalent to sign(a[j]) for non-zero
        mult_j = multiplicity[j]
        
        # Scaling factor from logic: multiWeightJ = a[j] / multiplicity[j]
        # We need this for the wjk calc
        val_j_scaled = val_j / mult_j
        
        for rep in 1:mult_j
            # O(1) Sampling
            k = sample(wv)
            
            # We guaranteed k != i via weights. We only check k != j.
            if (k != j) & (j != i) & (k != i)
                val_k = a[k]
                abs_val_k = abs_a[k]
                
                # Calculate Weight wjk
                # wjk = abs(multiWeightJ * a[k]) / (|a[j]| + |a[k]|)
                wjk = abs(val_j_scaled * val_k) / (abs_val_j + abs_val_k)
                
                # Update Edge Counts
                @inbounds new_multiedges[j, k] += 1
                @inbounds new_multiedges[k, j] += 1
                
                # 7. Rank-1 Matrix Update (Scalar optimization)
                # Instead of L += w * b * b', we update 4 indices manually.
                # b[j] = 1, b[k] = -sign(a[k])/sign(a[j])
                s = -(val_k / abs_val_k) / sign_j # s is the value at b[k]
                
                # The outer product adds:
                # [j,j] -> |1|^2 * w      [j,k] -> 1 * conj(s) * w
                # [k,j] -> s * 1 * w      [k,k] -> |s|^2 * w
                
                # Note: |s| = 1 for complex sign, so |s|^2 = 1.
                
                @inbounds begin
                    cliqueSampleLap[j, j] += wjk
                    cliqueSampleLap[k, k] += wjk
                    cliqueSampleLap[j, k] += wjk * conj(s)
                    cliqueSampleLap[k, j] += wjk * s
                end
            end
        end
    end
    
    return cliqueSampleLap, new_multiedges
end

In [16]:
function multi_erase_self_loops(A::AbstractMatrix, B::AbstractMatrix, IncidenceA, IncidenceB)
    m, n = size(A)
    @assert size(B) == (m, n) "Matrices A and B must have the same dimensions"

    # Initialize C with zeros (efficient allocation)
    # Use the same type as the input matrices (e.g., Float64)
    C = zeros(promote_type(eltype(A), eltype(B)), m, n)
    IncidenceNew= IncidenceA+IncidenceB
    @inbounds for j in 1:n
        for i in 1:j
            valA = A[i, j]
            valB = B[i, j]
            if (abs(valA) >1e-12) & (abs(valB) >1e-12)               
                # 1. Base assignment
                C[i, j] = valA + valB
                C[j, i] = valA + valB
                # 2. Calculate the penalty term
                # Term = 2 * (|A| + |B| - |A+B|)
                #term = (abs(valA) + abs(valB) - abs(valA + valB))
                #println(term)
                # 3. Add to diagonals
                C[i, i] += abs(valA) + abs(valB)
                C[j, j] += abs(valA) + abs(valB) 
            end
            if (abs(valA) >1e-12) & (abs(valB) <=1e-12)
                C[i,j] = valA
                C[j,i] = valA
                C[i, i] += abs(valA) 
                C[j, j] += abs(valA) 
                
            end
            if (abs(valA) <=1e-12) & (abs(valB) <=1e-12)
                C[i,j] = valB
                C[j,i] = valB
                C[i, i] += abs(valB)
                C[j, j] += abs(valB) 
            end
            # If signs are the same, "nothing happens" (C[i,j] remains 0)
        end
    end

    return C, IncidenceNew
end

multi_erase_self_loops (generic function with 1 method)

In [17]:
# We need a datastructure that holds the number of multiedges, this is the most naive way to do it
# Note that multiedges == 1 is not equal to no multiedges.
function initMultiedges(L, nr_multi)
  multiedges = zeros(Int, n,n,)
  A_mask = abs.(-L + Diagonal(diag(L))).> 0
  multiedges[A_mask] .= nr_multi
  return multiedges
end

initMultiedges (generic function with 1 method)

In [ ]:
# This is still the same algorithm and some index handling
function apxCholMultiHDD(L, nr_multi)
  multi_edges = initMultiedges(L, nr_multi)
  n = size(L)[1]
  S = copy(L)
  fac = zeros(ComplexF64,n,n)
  for i = 1:(n-1)

    li = S[:,i]/S[i,i]^(1/2)
    fac[:,i] = li

    ai = S[:,i]
    di = S[i,i]

    # S without star on i
    S_noi = copy(S)
    S_noi -=Diagonal(abs.(ai))
    S_noi[:,i] .= 0
    S_noi[i,:] .= 0
    sampled_clique, new_multiedges = cliqueSampleMultiEdgeSddEff(ai, multi_edges[:,i],i)
    S = S_noi + sampled_clique
    multi_edges += new_multiedges
    if i==n-1
      fac[n,n] = sqrt(abs(S_noi[n,n]+ sampled_clique[n,n]))
    end
  end
  return fac
end

apxCholMultiSDD (generic function with 1 method)

In [ ]:
# This is still the same algorithm and some index handling
function apxCholMultiHDD(L, nr_multi)
  multi_edges = initMultiedges(L, nr_multi)
  n = size(L)[1]
  S = copy(L)
  fac = zeros(ComplexF64,n,n)
  for i = 1:(n-1)

    li = S[:,i]/S[i,i]^(1/2)
    fac[:,i] = li

    ai = S[:,i]
    di = S[i,i]

    # S without star on i
    S_noi = copy(S)
    S_noi -=Diagonal(abs.(ai))
    S_noi[:,i] .= 0
    S_noi[i,:] .= 0
    sampled_clique, new_multiedges = cliqueSampleMultiEdgeSddEff(ai, multi_edges[:,i],i)
    S = S_noi + sampled_clique
    multi_edges += new_multiedges
    if i==n-1
      fac[n,n] = sqrt(abs(S_noi[n,n]+ sampled_clique[n,n]))
    end
  end
  return fac
end

In [20]:
# This is still the same algorithm and some index handling
function apxCholMulti(L, nr_multi)
  multi_edges = initMultiedges(L, nr_multi)
  n = size(L)[1]
  S = copy(L)
  fac = zeros(n,n-1)
  for i = 1:(n-1)
    li = S[:,i]/S[i,i]^(1/2)
    fac[:,i] = li

    ai = -S[:,i]
    di = S[i,i]
    ai[i] = 0

    # S without star on i
    S_noi = copy(S)
    S_noi[:,i] .= 0
    S_noi[i,:] .= 0
    S_noi -= Diagonal(ai)
    sampled_clique, new_multiedges = cliqueSampleMultiEdge(ai, multi_edges[:,i])
    S = S_noi + sampled_clique
    multi_edges += new_multiedges
  end
  return fac
end

apxCholMulti (generic function with 1 method)

In [21]:
# Same example
n = 10
L_Kn = I - ones(n,n)/n

fApxChol = apxCholMulti(L_Kn, 10000) # Second param specifies the number of multiedges
rel_eigs = eigvals((L_Kn + ones(n,n) / n)*inv(fApxChol*fApxChol' + ones(n,n) / n))

# The ratio of the smallest to the largest eigenvalue
maximum(rel_eigs) / minimum(rel_eigs)

1.0054371464625147

In [22]:
n = 10
L_Kn = I - ones(n,n)/n
#L_Kn[1,2]= -L_Kn[1,2]
#L_Kn[2,1]= -L_Kn[2,1]

fApxChol = apxCholMultiSDD(L_Kn, 100000) # Second param specifies the number of multiedges
rel_eigs = eigvals((L_Kn + ones(n,n) / n)*inv(fApxChol*fApxChol' + ones(n,n) / n))

# The ratio of the smallest to the largest eigenvalue
maximum(rel_eigs) / minimum(rel_eigs)

1.0016607284612749

In [23]:
function random_sdd_matrix(n::Int; strict::Bool=true)
    # 1. Generate a random n x n matrix
    A = randn(n, n)
    
    # 2. Make it symmetric by adding it to its transpose
    # This ensures A[i,j] == A[j,i]
    A = A + A'
    
    # 3. Calculate the sum of absolute values of off-diagonal elements
    # sum(abs, A, dims=2) sums the absolute values of each row.
    # We subtract the absolute value of the current diagonal to get just the off-diagonals.
    off_diag_sums = vec(sum(abs, A, dims=2)) .- abs.(diag(A))
    
    # 4. Create new diagonal values
    # To satisfy dominance, the diagonal must be >= off_diag_sums.
    # We add a small random 'padding' to make it strictly dominant (invertible).
    padding = strict ? (rand(n) .+ 0.1) : zeros(n)
    new_diag = off_diag_sums .+ padding
    
    # 5. Apply the new diagonal to A
    # We use diagind(A) for efficient in-place modification
    A[diagind(A)] = new_diag
    
    return A
end

random_sdd_matrix (generic function with 1 method)

In [ ]:
for j in 1:100
  n = 10
  S = random_sdd_matrix(n)
  fApxChol = apxCholMultiSDD(S, 10000) # Second param specifies the number of multiedges
  #print(size(fApxChol))
  rel_eigs = eigvals(S-fApxChol*fApxChol')

  # The ratio of the smallest to the largest eigenvalue

  println(maximum(rel_eigs) / minimum( abs(x)  for x in eigvals(S) if x>0.000001))

In [25]:
fApxChol = apxCholMultiSDD(S, 1000) # Second param specifies the number of multiedges
print(size(fApxChol))
rel_eigs = eigvals(S*pinv(fApxChol*fApxChol'))

# The ratio of the smallest to the largest eigenvalue

maximum(rel_eigs) / minimum(abs(x) for x  in rel_eigs if  x> 0.00000001)

(10, 9)

1.4400677889715532

In [26]:
# Now the best sampling rule, where we always get a tree
function cliqueSampleTree(a)
  d = sum(a)
  a_c = copy(a)
  nbrs = findall(a .> 1e-12)
  cliqueSampleLap = zeros(n,n)
  for j in nbrs[1:end-1]
    a_c[j] = 0
    k = sample(1:n,Weights(a_c))
    if k != j
      wjk = sum(a_c)*a[j] / d
      bjk = zeros(n); bjk[j] = 1; bjk[k] = -1;
      cliqueSampleLap += wjk*bjk*bjk'
    end
  end
return cliqueSampleLap
end

cliqueSampleTree (generic function with 1 method)

In [27]:
function apxCholTree(L)
  n = size(L)[1]
  S = copy(L)
  fac = zeros(n,n-1)
  for i = 1:(n-1)
    li = S[:,i]/S[i,i]^(1/2)
    fac[:,i] = li

    # Remove the star...
    ai = -S[:,i]
    di = S[i,i]
    ai[i] = 0

    S_noi = copy(S)
    S_noi[:,i] .= 0
    S_noi[i,:] .= 0
    S_noi -= Diagonal(ai)

    S = S_noi + cliqueSampleTree(ai)
  end
  return fac
end

apxCholTree (generic function with 1 method)

In [28]:
# Same example
n = 20
L_Kn = I - ones(n,n)/n

fApxChol = apxCholTree(L_Kn)
rel_eigs = eigvals((L_Kn + ones(n,n) / n)*inv(fApxChol*fApxChol' + ones(n,n) / n))

# The ratio of the smallest to the largest eigenvalue
println(maximum(rel_eigs) / minimum(rel_eigs))


2.3935893541893383


On these small graphs all versions (cliqueSimple are decent, lets make the graphs more challenging, this is a barbell graph with a long path


In [29]:
B = 10
n = 3*B
A_barbell = zeros(n,n)
A_barbell[1:B,1:B] = ones(B,B) / B
A_barbell[2B+1:3B,2B+1:3B] = ones(B,B) / B

A_barbell[B+1:2B, B+1:2B] += diagm(-1 => ones(B-1))
A_barbell[B+1:2B, B+1:2B] += diagm(1 => ones(B-1))
A_barbell[B+1,B] = 1
A_barbell[B,B+1] = 1
A_barbell[2B+1,2B] = 1
A_barbell[2B,2B+1] = 1
L_barbell = -A_barbell + Diagonal(vec(sum(A_barbell,dims=1)))
L_barbell



30×30 Matrix{Float64}:
  0.9  -0.1  -0.1  -0.1  -0.1  -0.1  …   0.0   0.0   0.0   0.0   0.0   0.0
 -0.1   0.9  -0.1  -0.1  -0.1  -0.1      0.0   0.0   0.0   0.0   0.0   0.0
 -0.1  -0.1   0.9  -0.1  -0.1  -0.1      0.0   0.0   0.0   0.0   0.0   0.0
 -0.1  -0.1  -0.1   0.9  -0.1  -0.1      0.0   0.0   0.0   0.0   0.0   0.0
 -0.1  -0.1  -0.1  -0.1   0.9  -0.1      0.0   0.0   0.0   0.0   0.0   0.0
 -0.1  -0.1  -0.1  -0.1  -0.1   0.9  …   0.0   0.0   0.0   0.0   0.0   0.0
 -0.1  -0.1  -0.1  -0.1  -0.1  -0.1      0.0   0.0   0.0   0.0   0.0   0.0
 -0.1  -0.1  -0.1  -0.1  -0.1  -0.1      0.0   0.0   0.0   0.0   0.0   0.0
 -0.1  -0.1  -0.1  -0.1  -0.1  -0.1      0.0   0.0   0.0   0.0   0.0   0.0
 -0.1  -0.1  -0.1  -0.1  -0.1  -0.1      0.0   0.0   0.0   0.0   0.0   0.0
  ⋮                             ⋮    ⋱         ⋮                      
  0.0   0.0   0.0   0.0   0.0   0.0     -0.1  -0.1  -0.1  -0.1  -0.1  -0.1
  0.0   0.0   0.0   0.0   0.0   0.0     -0.1  -0.1  -0.1  -0.1  -0.1  -0.1
  0.0 

In [30]:
# Here the iid sampling
fApxChol_iid = apxChol1(L_barbell)
rel_eigs_iid = eigvals(Symmetric((L_barbell + ones(n,n) / n)*inv(fApxChol_iid*fApxChol_iid' + ones(n,n) / n)))


# The ratio of the smallest to the largest eigenvalue
maximum(rel_eigs_iid) / minimum(rel_eigs_iid)


6.652963071898838

In [31]:
# Here the Tree version
fApxChol_tree = apxCholTree(L_barbell)
rel_eigs_tree = eigvals(Symmetric((L_barbell + ones(n,n) / n)*inv(fApxChol_tree*fApxChol_tree' + ones(n,n) / n)))

maximum(rel_eigs_tree) / minimum(rel_eigs_tree)

2.969775258539018

In [32]:
# We still don't really break the easy version. Here a star graph, which should be very hard for the easy version
n = 10
L_star = zeros(n,n)+ I
L_star[1,:] .= -1
L_star[:,1] .= -1
L_star[1,1] = n-1
L_star

10×10 Matrix{Float64}:
  9.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0
 -1.0   1.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0
 -1.0   0.0   1.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0
 -1.0   0.0   0.0   1.0   0.0   0.0   0.0   0.0   0.0   0.0
 -1.0   0.0   0.0   0.0   1.0   0.0   0.0   0.0   0.0   0.0
 -1.0   0.0   0.0   0.0   0.0   1.0   0.0   0.0   0.0   0.0
 -1.0   0.0   0.0   0.0   0.0   0.0   1.0   0.0   0.0   0.0
 -1.0   0.0   0.0   0.0   0.0   0.0   0.0   1.0   0.0   0.0
 -1.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   1.0   0.0
 -1.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   1.0

In [33]:
# Here the iid sampling
repeats = 10
for i in 1:repeats
  fApxChol_iid = apxChol1(L_star)
  rel_eigs_iid = eigvals(Symmetric((L_star + ones(n,n) / n)*pinv(fApxChol_iid*fApxChol_iid' + ones(n,n) / n)))


  # The ratio of the smallest to the largest eigenvalue
  println("Run id: ", i, " ", maximum(rel_eigs_iid) / minimum(rel_eigs_iid))
end

The factorization disconnected the graph, anything can happen now!
Run id: 1 9.667928998017078e15
Run id: 2 45.64407651984126
The factorization disconnected the graph, anything can happen now!
Run id: 3 4.114457440501168e16
The factorization disconnected the graph, anything can happen now!
Run id: 4 -9.17396187815181e15
The factorization disconnected the graph, anything can happen now!
Run id: 5 -1.94344656052972e16
The factorization disconnected the graph, anything can happen now!
Run id: 6 -2.0207548915548652e16
The factorization disconnected the graph, anything can happen now!
Run id: 7 -2.5831833034746005e15
The factorization disconnected the graph, anything can happen now!
Run id: 8 -6.425195268766145e15
The factorization disconnected the graph, anything can happen now!
Run id: 9 -1.4209939974999596e16
Run id: 10 10.864022428173058


In [34]:
# Here the Tree version
repeats = 10
for i in 1:repeats
  fApxChol_tree = apxCholTree(L_star)
  rel_eigs_tree = eigvals(Symmetric((L_star + ones(n,n) / n)*inv(fApxChol_tree*fApxChol_tree' + ones(n,n) / n)))
  println("Run id: ", i, " ", maximum(rel_eigs_tree) / minimum(rel_eigs_tree))
end


Run id: 1 47.30058336716477
Run id: 2 60.30498491876688
Run id: 3 56.97268368646562
Run id: 4 27.262367926391093
Run id: 5 70.36972748059344
Run id: 6 48.677926734679126
Run id: 7 68.29785928039095
Run id: 8 37.49425597617248
Run id: 9 85.16591312545684
Run id: 10 91.74845764673977


# Final words
Of course this naive version should not be used in practice. We have now optimized it for understandability. Instead, there is a fast implementation using sparse datastructures and other optimizations. You can access the serious implementation here:

[Github link: ApproxChol SDD solver](https://rjkyng.github.io/SDDM2023/)

This also includes a multi-edge version of cliqueSampleTree, but, the single-sample version is generally enough.

If you have any questions about this code or the serious implementation you can send them to

ybaumann@ethz.ch
and/or
kyng@inf.ethz.ch 



